# Binary Random Forest Classification — Feature Configuration I

Trains and evaluates a Random Forest classifier to separate allergenic from
non-allergenic tree crowns (binary classification).

**Feature Configuration I** uses spectral and structural features only
(R, G, B, NIR mean/std; CHM mean/std/max; crown area). NDVI-derived features
are excluded. This configuration serves as the baseline for comparison with
Feature Configuration II.

**Workflow**
1. Load crown feature shapefile from `rf_binary/enschede_rf_binary_features_v01.shp`.
2. Split into labeled (public trees with species labels) and unlabeled
   (private trees, no labels).
3. Prepare feature matrix — NDVI columns excluded.
4. Train/test split (80/20, stratified).
5. GridSearchCV with 10-fold cross-validation to find optimal hyperparameters.
6. Evaluate on the test set (accuracy, AUC-ROC, confusion matrix, precision/recall).
7. Compute SHAP feature importance.
8. Predict allergenicity of private trees; apply 0.6 probability threshold.
9. Export filtered predictions to `rf_binary/enschede_rf_binary_predictions_fc1_v01.shp`.

**Import libraries**

In [ ]:
import time
import sys
from collections import Counter
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, train_test_split

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

from src.paths import PROCESSED_DIR

**Define paths**

In [ ]:
# Crown polygons with extracted spectral and structural features
TREE_FEATURES_DIR = PROCESSED_DIR / "rf_binary/enschede_rf_binary_features_v01.shp"

# Output path for filtered binary predictions
OUTPUT_PATH = PROCESSED_DIR / "rf_binary/enschede_rf_binary_predictions_fc1_v01.shp"

**Define constants**

In [ ]:
LABEL_MAP = {0.0: "Non-allergenic", 1.0: "Allergenic"}

# Columns excluded from features: identifiers, target, geometry, and NDVI
# (NDVI is excluded in FC1; included in FC2)
DROP_COLS_FC1 = ["id", "species_na", "is_allerge", "species_id",
                 "geometry", "NDVI_mean", "NDVI_std"]

**Load data**

In [ ]:
trees_df = gpd.read_file(TREE_FEATURES_DIR)
print(f"Total crowns: {len(trees_df)}")
print(trees_df["is_allerge"].value_counts(dropna=False))

**Split into labeled (public) and unlabeled (private) trees**

In [ ]:
# Public trees have a ground-truth allergenicity label from the municipal inventory.
# Private trees have NaN and are used exclusively for prediction.
labeled_df   = trees_df[trees_df["is_allerge"].notna()].copy()
unlabeled_df = trees_df[trees_df["is_allerge"].isna()].copy()

print(f"Labeled (public):   {len(labeled_df)}")
print(f"Unlabeled (private):{len(unlabeled_df)}")

**Prepare feature matrices**

In [ ]:
X_labeled   = labeled_df.drop(columns=DROP_COLS_FC1)
X_unlabeled = unlabeled_df.drop(columns=DROP_COLS_FC1)
y_labeled   = labeled_df["is_allerge"]

print("X labeled shape:  ", X_labeled.shape)
print("X unlabeled shape:", X_unlabeled.shape)
print("Feature columns:  ", list(X_labeled.columns))

**Train/test split**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_labeled, y_labeled,
    test_size=0.2,
    random_state=42,
    stratify=y_labeled,
)

**Class distribution per split**

In [ ]:
def plot_class_distribution(y_train, y_test, label_map):
    """
    Plot a bar chart showing class sample counts across train and test splits.

    Args:
        y_train:   Training target series.
        y_test:    Test target series.
        label_map: Dict mapping numeric label to class name string.
    """
    splits = {"Train": Counter(y_train.map(label_map)),
              "Test":  Counter(y_test.map(label_map))}
    classes = list(label_map.values())
    colors  = plt.cm.Set2(np.linspace(0, 1, len(classes)))

    fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=120)
    for ax, (split, counts) in zip(axes, splits.items()):
        vals = [counts.get(c, 0) for c in classes]
        bars = ax.bar(classes, vals, color=colors, edgecolor="black")
        ax.set_title(split, fontsize=14, weight="bold")
        ax.set_xlabel("Class")
        ax.set_ylabel("Number of samples")
        ax.set_xticklabels(classes, rotation=30, ha="right", fontsize=10)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + max(vals) * 0.01,
                    str(v), ha="center", va="bottom", fontsize=9, weight="bold")
    fig.suptitle("Class samples per split", fontsize=16, weight="bold", y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_class_distribution(y_train, y_test, LABEL_MAP)

**Train Random Forest with GridSearchCV**

In [ ]:
def train_rf(X_train, y_train, param_grid, label=""):
    """
    Train a Random Forest classifier using 10-fold GridSearchCV.

    Balanced class weights are applied to compensate for class imbalance.
    n_jobs=1 is used to ensure reproducibility and avoid memory issues on
    the CRIB computing platform.

    Args:
        X_train:    Training feature DataFrame.
        y_train:    Training target series.
        param_grid: Dict of hyperparameter ranges for GridSearchCV.
        label:      Optional label printed in the output for identification.

    Returns:
        The best fitted RandomForestClassifier from GridSearchCV.
    """
    classifier = RandomForestClassifier(
        random_state=42,
        n_jobs=1,
        class_weight="balanced",
    )
    grid = GridSearchCV(
        classifier,
        param_grid,
        cv=10,
        scoring="accuracy",
        n_jobs=1,
        verbose=1,
    )
    start = time.time()
    grid.fit(X_train, y_train)
    print(f"{label} Best parameters: {grid.best_params_}")
    print(f"{label} Best CV accuracy: {grid.best_score_:.4f}")
    print(f"{label} Training time:    {time.time() - start:.0f} s")
    return grid.best_estimator_

In [ ]:
param_grid = {
    "n_estimators": [200, 500, 800],
    "max_depth":    [None, 10, 20],
}

best_model = train_rf(X_train, y_train, param_grid, label="[FC1 Binary]")

**Evaluate on test set**

In [ ]:
def evaluate_binary(model, X_test, y_test):
    """
    Evaluate a binary Random Forest classifier on the test set.

    Prints accuracy, AUC-ROC, and a classification report.
    Plots a normalised confusion matrix and a per-class precision/recall bar chart.

    The confusion matrix is transposed so that rows = predicted and columns = true,
    matching the orientation used in the YOLO evaluation notebooks.

    Args:
        model:  Fitted RandomForestClassifier.
        X_test: Test feature DataFrame.
        y_test: Test target series.
    """
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    print("Test accuracy:", accuracy_score(y_test, y_pred))
    print("AUC-ROC:      ", roc_auc_score(y_test, y_prob))
    print()
    print(classification_report(y_test, y_pred))

    labels = ["Non-allergenic", "Allergenic"]

    # Normalised confusion matrix (transposed: rows = predicted, cols = true)
    cm = confusion_matrix(y_test, y_pred).T
    cm_norm = cm.astype("float") / cm.sum(axis=0)

    plt.figure(figsize=(6, 5))
    sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=labels, yticklabels=labels, vmin=0, vmax=1)
    plt.title("Normalised Confusion Matrix")
    plt.xlabel("True")
    plt.ylabel("Predicted")
    plt.tight_layout()
    plt.show()

    # Per-class precision / recall bar chart
    report = classification_report(y_test, y_pred, output_dict=True)
    df_m = pd.DataFrame({
        "Class":     labels,
        "Precision": [report["0.0"]["precision"], report["1.0"]["precision"]],
        "Recall":    [report["0.0"]["recall"],    report["1.0"]["recall"]],
    })
    x = np.arange(len(df_m))
    bw = 0.35
    fig, ax = plt.subplots(figsize=(8, 5))
    for i, metric in enumerate(["Precision", "Recall"]):
        ax.bar(x + i * bw, df_m[metric], width=bw, label=metric)
    ax.set_xticks(x + bw / 2)
    ax.set_xticklabels(df_m["Class"])
    ax.set_ylabel("Metric value")
    ax.set_title("Per-Class Precision and Recall")
    ax.legend()
    ax.grid(axis="y", linestyle="--", alpha=0.6)
    fig.tight_layout()
    plt.show()

In [ ]:
evaluate_binary(best_model, X_test, y_test)

**SHAP feature importance**

In [ ]:
def shap_analysis_binary(model, X_train, feature_names, n_samples=500):
    """
    Compute and plot SHAP feature importance for the allergenic class.

    A random subsample of the training set is used to keep runtime manageable.
    Mean absolute SHAP values are plotted as a horizontal bar chart for the
    top 15 features.

    Args:
        model:         Fitted RandomForestClassifier.
        X_train:       Training feature DataFrame.
        feature_names: List of feature column names.
        n_samples:     Number of training samples to use for SHAP computation.
    """
    X_df     = pd.DataFrame(X_train, columns=feature_names)
    X_sample = X_df.sample(n_samples, random_state=42)

    explainer   = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_sample)
    shap_cls    = shap_values[1]  # allergenic class

    shap_table = pd.DataFrame({
        "feature":      feature_names,
        "mean_abs_shap": np.abs(shap_cls).mean(axis=0),
    }).sort_values("mean_abs_shap", ascending=False)

    print("Top 20 features by SHAP importance:")
    print(shap_table.head(20).to_string(index=False))

    plot_data = shap_table.head(15).sort_values("mean_abs_shap", ascending=True)
    fig, ax   = plt.subplots(figsize=(12, 9))
    colors    = sns.color_palette("viridis", len(plot_data))
    bars      = ax.barh(plot_data["feature"], plot_data["mean_abs_shap"],
                        color=colors, edgecolor="black", alpha=0.8)
    ax.bar_label(bars,
                 labels=[f" {f}: {v:.4f}" for f, v in
                         zip(plot_data["feature"], plot_data["mean_abs_shap"])],
                 padding=5, fontsize=10, fontweight="bold")
    ax.set_title("Top 15 Features by SHAP Importance — Allergenic Class",
                 fontsize=16, pad=20)
    ax.set_xlabel("Mean Absolute SHAP Value", fontsize=12)
    ax.set_ylabel("Feature", fontsize=12)
    ax.set_xlim(0, plot_data["mean_abs_shap"].max() * 1.3)
    ax.grid(axis="x", linestyle="--", alpha=0.4)
    sns.despine(left=True, bottom=True)
    plt.tight_layout()
    plt.show()

In [ ]:
shap_analysis_binary(best_model, X_train, list(X_labeled.columns))

**Predict allergenicity of private trees**

In [ ]:
# Predict allergenicity for private trees (unlabeled dataset).
unlabeled_df["pred_is_allergenic"]   = best_model.predict(X_unlabeled)
unlabeled_df["pred_prob_allergenic"] = best_model.predict_proba(X_unlabeled)[:, 1]

print("Private tree prediction distribution:")
print(unlabeled_df["pred_is_allergenic"].value_counts(normalize=True).round(3))

**Filter and export predictions**

In [ ]:
# Write binary predictions back onto the full GeoDataFrame
trees_df.loc[trees_df["is_allerge"].isna(), "pred_is_allergenic"]  = (
    unlabeled_df["pred_is_allergenic"].values)
trees_df.loc[trees_df["is_allerge"].isna(), "pred_prob_allergenic"] = (
    unlabeled_df["pred_prob_allergenic"].values)

# Keep:
#   (a) all public trees confirmed as allergenic (is_allerge == 1)
#   (b) private trees predicted as allergenic with confidence >= 0.6
#
# The 0.6 threshold excludes borderline predictions where the model is
# uncertain (near 0.5), retaining only reasonably confident predictions
# for forwarding to multi-class classification.
trees_filtered = trees_df[
    (trees_df["is_allerge"] == 1) |
    (
        trees_df["is_allerge"].isna() &
        (trees_df["pred_is_allergenic"] == 1) &
        (trees_df["pred_prob_allergenic"] >= 0.6)
    )
].copy()

print(f"Total retained: {len(trees_filtered)}")
print(f"  Public allergenic: {(trees_filtered['is_allerge'] == 1).sum()}")
print(f"  Private predicted: {trees_filtered['is_allerge'].isna().sum()}")

trees_filtered.to_file(OUTPUT_PATH)
print(f"Saved: {OUTPUT_PATH}")